Understand duplicate execution

The same employee's two-day leave request gets created twice.

This is the problem we're trying to solve.

In [0]:
leave_requests = []

def apply_leave(employee_id, leave_days):
    leave_requests.append({
        "employee_id": employee_id,
        "leave_days": leave_days
    })

apply_leave("EMP101", 2)
apply_leave("EMP101", 2)

print(leave_requests)

Make the function idempotent

We'll introduce a unique request_id.

The idea is simple: if the same request ID is submitted again, don't create another request.

In [0]:
leave_requests = []

def apply_leave(request_id, employee_id, leave_days):

    for request in leave_requests:
        if request["request_id"] == request_id:
            return "Request already exists!"

    leave_requests.append({
        "request_id": request_id,
        "employee_id": employee_id,
        "leave_days": leave_days
    })

    return "Leave request created successfully!"


print(apply_leave("REQ101", "EMP101", 2))
print(apply_leave("REQ101", "EMP101", 2))

print(leave_requests)

It acts as a unique identifier for an operation.

Even if the agent retries the same request multiple times, the application recognises it and prevents duplicate creation.

In [0]:
print(apply_leave("REQ102", "EMP101", 3))
print(leave_requests)

Same request ID + repeated execution → No duplicate.

Different request ID → New request is allowed.

BELOW WE DO Tool safety

Authorisation means checking whether a user has permission to perform a particular action.

In [0]:
def approve_leave(employee_id, approver_role):

    if approver_role != "HR_MANAGER":
        return "Access denied! You are not authorised."

    return f"Leave approved for {employee_id}"


print(approve_leave("EMP101", "EMPLOYEE"))
print(approve_leave("EMP101", "HR_MANAGER"))

What did we implement?

EMPLOYEE → Cannot approve leave.

HR_MANAGER → Can approve leave.

## We'll combine input validation, authorisation and idempotency into a safer HR tool, then connect these ideas to our Databricks agent.

In [0]:
approved_requests = set()

def approve_leave(request_id, employee_id, leave_days, approver_role):

    # 1. Input validation
    if not employee_id.startswith("EMP"):
        return "Invalid employee ID"

    if leave_days <= 0:
        return "Invalid leave days"

    # 2. Authorisation
    if approver_role != "HR_MANAGER":
        return "Access denied"

    # 3. Idempotency
    if request_id in approved_requests:
        return "Request already approved"

    approved_requests.add(request_id)

    return f"Leave approved for {employee_id}"


print(approve_leave("REQ101", "EMP101", 2, "HR_MANAGER"))
print(approve_leave("REQ101", "EMP101", 2, "HR_MANAGER"))
print(approve_leave("REQ102", "EMP101", -2, "HR_MANAGER"))
print(approve_leave("REQ103", "EMP101", 2, "EMPLOYEE"))